# 📊 Statistical Analysis — So sánh 3 LLM sinh Test Case
**Đề tài:** Phân tích hiệu quả của LLM trong quy trình phát triển phần mềm

Notebook này thực hiện:
1. Nhập điểm từ Evaluation Sheet
2. Thống kê mô tả (mean, std, median, min/max)
3. Kiểm định t-test (paired) & Wilcoxon signed-rank test
4. Effect size — Cohen's d
5. Radar chart so sánh 4 dimension
6. Bar chart + box plot tổng hợp
7. Kết luận tự động theo ngưỡng p-value

## Cell 1 — Cài đặt thư viện

In [ ]:
!pip install -q scipy numpy pandas matplotlib seaborn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import wilcoxon, ttest_rel, shapiro
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 150
sns.set_theme(style='whitegrid', palette='muted')
print('✅ Thư viện sẵn sàng!')

## Cell 2 — ✏️ NHẬP DỮ LIỆU
> **Điền điểm từ Evaluation Sheet vào đây.** Mỗi list = điểm của 1 LLM trên từng feature.
> Điểm theo thang 10, 4 dimension theo thang tối đa tương ứng (3-3-2-2).

In [ ]:
# ============================================================
# ✏️  ĐIỀN DỮ LIỆU TỪ EVALUATION SHEET VÀO ĐÂY
# ============================================================

# Tên các feature đã test
FEATURES = [
    'Login',
    'Search',
    'Booking Flow',
    # Thêm feature nếu có
]

# ── ĐIỂM TỔNG (0-10) ────────────────────────────────────────
# Mỗi phần tử = điểm của 1 feature
scores = {
    'ChatGPT':  [8.5, 7.8, 8.2],   # ← điền điểm thực tế
    'Gemini':   [7.2, 6.9, 7.5],
    'Qwen':     [5.8, 5.2, 6.0],
}

# ── ĐIỂM TỪNG DIMENSION ─────────────────────────────────────
# dimension_scores[model][feature] = [d1, d2, d3, d4]
# d1: Coverage (0-3), d2: Writing Quality (0-3),
# d3: Format (0-2),   d4: Domain & Logic (0-2)
dimension_scores = {
    'ChatGPT': [
        [2.5, 2.5, 2.0, 1.5],  # Feature 1
        [2.2, 2.3, 1.8, 1.5],  # Feature 2
        [2.4, 2.4, 2.0, 1.4],  # Feature 3
    ],
    'Gemini': [
        [2.0, 2.2, 1.8, 1.2],
        [2.0, 2.0, 1.6, 1.3],
        [2.2, 2.1, 1.8, 1.4],
    ],
    'Qwen': [
        [1.8, 1.5, 1.5, 1.0],
        [1.5, 1.5, 1.2, 1.0],
        [2.0, 1.6, 1.4, 1.0],
    ],
}

# ── METRICS PHỤ ─────────────────────────────────────────────
# Số TC sinh ra trung bình mỗi feature
tc_counts = {
    'ChatGPT': [18, 22, 25],
    'Gemini':  [15, 18, 20],
    'Qwen':    [12, 14, 16],
}
# Thời gian sinh (giây)
gen_time = {
    'ChatGPT': [12, 15, 18],   # API call
    'Gemini':  [8,  10, 12],
    'Qwen':    [45, 52, 60],   # Local inference
}
# Chi phí ước tính USD
cost = {
    'ChatGPT': [0.04, 0.05, 0.06],
    'Gemini':  [0.00, 0.00, 0.00],
    'Qwen':    [0.00, 0.00, 0.00],
}

# ============================================================
# Chuyển sang DataFrame
df = pd.DataFrame(scores, index=FEATURES)
print('✅ Dữ liệu đã load:')
print(df)

## Cell 3 — Thống kê mô tả

In [ ]:
print('=' * 65)
print('  THỐNG KÊ MÔ TẢ — ĐIỂM CHẤT LƯỢNG TEST CASE (thang 10)')
print('=' * 65)

desc_rows = []
for model in scores:
    s = np.array(scores[model])
    desc_rows.append({
        'Model': model,
        'Mean': round(s.mean(), 3),
        'Std Dev': round(s.std(ddof=1), 3),
        'Median': round(np.median(s), 3),
        'Min': s.min(),
        'Max': s.max(),
        'Range': round(s.max() - s.min(), 3),
    })

desc_df = pd.DataFrame(desc_rows).set_index('Model')
print(desc_df.to_string())

print()
best = desc_df['Mean'].idxmax()
print(f'  → Model có điểm trung bình cao nhất: {best} ({desc_df.loc[best,"Mean"]:.3f}/10)')

# Dimension averages
print()
print('-' * 65)
print('  ĐIỂM TRUNG BÌNH THEO DIMENSION')
print('-' * 65)
dim_labels = ['Coverage (max 3)', 'Writing Quality (max 3)',
              'Format (max 2)', 'Domain & Logic (max 2)']
dim_df_rows = []
for model in dimension_scores:
    arr = np.array(dimension_scores[model])  # shape: (n_features, 4)
    means = arr.mean(axis=0)
    row = {'Model': model}
    for i, lbl in enumerate(dim_labels):
        row[lbl] = round(means[i], 3)
    dim_df_rows.append(row)
dim_df = pd.DataFrame(dim_df_rows).set_index('Model')
print(dim_df.to_string())

## Cell 4 — Kiểm định phân phối chuẩn (Shapiro-Wilk)
> Bước này quyết định dùng t-test (parametric) hay Wilcoxon (non-parametric)

In [ ]:
print('=' * 65)
print('  KIỂM ĐỊNH SHAPIRO-WILK — PHÂN PHỐI CHUẨN')
print('  H0: dữ liệu phân phối chuẩn | α = 0.05')
print('=' * 65)

normality = {}
for model in scores:
    s = np.array(scores[model])
    if len(s) < 3:
        print(f'  {model}: cần ít nhất 3 mẫu để kiểm định — skip')
        normality[model] = None
        continue
    stat, p = shapiro(s)
    normal = p > 0.05
    normality[model] = normal
    verdict = '✅ Phân phối chuẩn' if normal else '⚠️  KHÔNG phân phối chuẩn'
    print(f'  {model:12s}: W={stat:.4f}, p={p:.4f}  →  {verdict}')

all_normal = all(v for v in normality.values() if v is not None)
print()
if all_normal:
    print('  → Dùng Paired t-test (parametric)')
else:
    print('  → Dùng Wilcoxon signed-rank test (non-parametric)')
print()
print('  Lưu ý: Với n nhỏ (< 30), Wilcoxon an toàn hơn bất kể kết quả Shapiro.')

## Cell 5 — Paired t-test & Wilcoxon signed-rank test

In [ ]:
def cohen_d(a, b):
    """Cohen's d — effect size cho paired samples."""
    diff = np.array(a) - np.array(b)
    return diff.mean() / (diff.std(ddof=1) + 1e-9)

def interpret_d(d):
    d = abs(d)
    if d < 0.2:   return 'negligible'
    elif d < 0.5: return 'small'
    elif d < 0.8: return 'medium'
    else:         return 'large'

def interpret_p(p, alpha=0.05):
    if p < 0.001:  return f'p={p:.4f} *** (highly significant)'
    elif p < 0.01: return f'p={p:.4f} **  (significant)'
    elif p < 0.05: return f'p={p:.4f} *   (significant)'
    else:          return f'p={p:.4f}     (not significant)'

models = list(scores.keys())
pairs = [(models[i], models[j])
         for i in range(len(models)) for j in range(i+1, len(models))]

results = []

print('=' * 75)
print('  KIỂM ĐỊNH THỐNG KÊ — SO SÁNH TỪNG CẶP LLM')
print('  α = 0.05 | Paired t-test + Wilcoxon signed-rank')
print('=' * 75)

for m1, m2 in pairs:
    a = np.array(scores[m1])
    b = np.array(scores[m2])
    diff = a - b

    # Paired t-test
    t_stat, t_p = ttest_rel(a, b)

    # Wilcoxon (nếu tất cả diff == 0 thì skip)
    if np.all(diff == 0):
        w_stat, w_p = np.nan, 1.0
    else:
        try:
            w_stat, w_p = wilcoxon(a, b, alternative='two-sided')
        except Exception:
            w_stat, w_p = np.nan, np.nan

    # Effect size
    d = cohen_d(a, b)

    # Mean difference
    mean_diff = diff.mean()

    results.append({
        'Pair': f'{m1} vs {m2}',
        'Mean diff': round(mean_diff, 3),
        't-stat': round(t_stat, 4),
        't p-value': round(t_p, 4),
        'W-stat': round(w_stat, 2) if not np.isnan(w_stat) else '-',
        'W p-value': round(w_p, 4) if not np.isnan(w_p) else '-',
        "Cohen's d": round(d, 3),
        'Effect': interpret_d(d),
    })

    print(f'\n  {m1}  vs  {m2}')
    print(f'  Mean difference   : {mean_diff:+.3f} (positive = {m1} cao hơn)')
    print(f'  Paired t-test     : t={t_stat:.4f},  {interpret_p(t_p)}')
    if not np.isnan(w_p):
        print(f'  Wilcoxon          : W={w_stat:.2f}, {interpret_p(w_p)}')
    print(f"  Cohen's d         : {d:.3f}  ({interpret_d(d)} effect)")

    # Kết luận
    sig = t_p < 0.05 or (not np.isnan(w_p) and w_p < 0.05)
    if sig and mean_diff > 0:
        print(f'  ✅ KẾT LUẬN: {m1} cao hơn {m2} có ý nghĩa thống kê')
    elif sig and mean_diff < 0:
        print(f'  ✅ KẾT LUẬN: {m2} cao hơn {m1} có ý nghĩa thống kê')
    else:
        print(f'  ⚠️  KẾT LUẬN: Không có sự khác biệt có ý nghĩa thống kê')

print('\n' + '=' * 75)
res_df = pd.DataFrame(results).set_index('Pair')
print('\nBẢNG TÓM TẮT:')
print(res_df.to_string())

## Cell 6 — Radar Chart (so sánh 4 dimension)

In [ ]:
# Chuẩn hóa dimension về % (so với điểm tối đa)
dim_max = [3, 3, 2, 2]   # max của từng dimension
dim_short = ['Coverage', 'Writing\nQuality', 'Format', 'Domain\n& Logic']
N = len(dim_short)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # đóng vòng

colors = {'ChatGPT': '#2196F3', 'Gemini': '#FF9800', 'Qwen': '#E91E63'}

fig, ax = plt.subplots(figsize=(7, 7),
                       subplot_kw=dict(polar=True))

for model in dimension_scores:
    arr = np.array(dimension_scores[model])
    means = arr.mean(axis=0)
    normalized = [means[i] / dim_max[i] * 10 for i in range(N)]
    normalized += normalized[:1]
    ax.plot(angles, normalized, 'o-', linewidth=2,
            label=model, color=colors[model])
    ax.fill(angles, normalized, alpha=0.15, color=colors[model])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(dim_short, size=12, fontweight='bold')
ax.set_ylim(0, 10)
ax.set_yticks([2, 4, 6, 8, 10])
ax.set_yticklabels(['2', '4', '6', '8', '10'], size=8, color='grey')
ax.grid(color='grey', linestyle='--', linewidth=0.5, alpha=0.7)

ax.set_title('Radar Chart — So sánh 4 Dimension\n(điểm chuẩn hóa 0-10)',
             size=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=11)

plt.tight_layout()
plt.savefig('radar_chart.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Lưu: radar_chart.png')

## Cell 7 — Bar Chart + Box Plot tổng hợp

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Phân tích so sánh 3 LLM — Sinh Test Case', fontsize=14,
             fontweight='bold', y=1.02)

palette = {'ChatGPT': '#2196F3', 'Gemini': '#FF9800', 'Qwen': '#E91E63'}

# ── Chart 1: Bar chart điểm TB mỗi feature ───────────────────
ax1 = axes[0]
x = np.arange(len(FEATURES))
width = 0.25
for i, (model, color) in enumerate(palette.items()):
    bars = ax1.bar(x + i*width, scores[model], width,
                   label=model, color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, scores[model]):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.1f}', ha='center', va='bottom', fontsize=8)

ax1.set_xlabel('Feature', fontsize=11)
ax1.set_ylabel('Điểm chất lượng (0-10)', fontsize=11)
ax1.set_title('Điểm theo Feature', fontsize=12, fontweight='bold')
ax1.set_xticks(x + width)
ax1.set_xticklabels(FEATURES, rotation=15, ha='right')
ax1.set_ylim(0, 11)
ax1.axhline(y=7, color='green', linestyle='--', alpha=0.5,
            label='Ngưỡng PASS (7.0)')
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)

# ── Chart 2: Box plot phân phối điểm ─────────────────────────
ax2 = axes[1]
box_data = [scores[m] for m in scores]
bp = ax2.boxplot(box_data, patch_artist=True, notch=False,
                 widths=0.5)
for patch, color in zip(bp['boxes'], palette.values()):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
for median in bp['medians']:
    median.set_color('black')
    median.set_linewidth(2)

# Overlay scatter
for i, (model, color) in enumerate(palette.items(), 1):
    y_vals = scores[model]
    x_vals = np.random.normal(i, 0.05, size=len(y_vals))
    ax2.scatter(x_vals, y_vals, color=color, zorder=5, s=50, alpha=0.9)

ax2.set_xticklabels(list(palette.keys()), fontsize=10)
ax2.set_ylabel('Điểm (0-10)', fontsize=11)
ax2.set_title('Phân phối điểm', fontsize=12, fontweight='bold')
ax2.axhline(y=7, color='green', linestyle='--', alpha=0.5)
ax2.grid(axis='y', alpha=0.3)

# ── Chart 3: Trade-off chi phí vs chất lượng ─────────────────
ax3 = axes[2]
mean_scores = {m: np.mean(scores[m]) for m in scores}
mean_cost = {m: np.mean(cost[m]) * 100 for m in cost}  # cents
mean_time = {m: np.mean(gen_time[m]) for m in gen_time}

for model, color in palette.items():
    ax3.scatter(mean_cost[model], mean_scores[model],
                s=mean_time[model] * 8,   # size = thời gian
                color=color, alpha=0.8, edgecolors='white', linewidth=1.5,
                zorder=5, label=model)
    ax3.annotate(f'{model}\n({mean_time[model]:.0f}s)',
                 (mean_cost[model], mean_scores[model]),
                 textcoords='offset points', xytext=(8, 5), fontsize=9)

ax3.set_xlabel('Chi phí trung bình (cents/feature)', fontsize=11)
ax3.set_ylabel('Điểm chất lượng TB (0-10)', fontsize=11)
ax3.set_title('Trade-off: Chi phí vs Chất lượng\n(kích thước bubble = thời gian sinh)',
              fontsize=11, fontweight='bold')
ax3.set_ylim(0, 10.5)
ax3.axhline(y=7, color='green', linestyle='--', alpha=0.5)
ax3.grid(alpha=0.3)
ax3.legend(fontsize=9)

plt.tight_layout()
plt.savefig('comparison_charts.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Lưu: comparison_charts.png')

## Cell 8 — Kết luận tự động & Tóm tắt cho báo cáo

In [ ]:
print('=' * 70)
print('  KẾT LUẬN THỐNG KÊ — TỰ ĐỘNG THEO DỮ LIỆU')
print('=' * 70)

mean_s = {m: np.mean(scores[m]) for m in scores}
best_model = max(mean_s, key=mean_s.get)
worst_model = min(mean_s, key=mean_s.get)
models_sorted = sorted(mean_s, key=mean_s.get, reverse=True)

print(f"""
1. RANKING TỔNG QUÁT
   {models_sorted[0]} > {models_sorted[1]} > {models_sorted[2]}
   Điểm: {mean_s[models_sorted[0]]:.2f} > {mean_s[models_sorted[1]]:.2f} > {mean_s[models_sorted[2]]:.2f} / 10
""")

# Check significance ChatGPT vs Qwen
a = np.array(scores['ChatGPT'])
b = np.array(scores['Qwen'])
_, p_cu = ttest_rel(a, b)
d_cu = cohen_d(a, b)

print(f"""2. SO SÁNH THƯƠNG MẠI vs OPEN-SOURCE
   ChatGPT vs Qwen: mean diff = {(a-b).mean():+.2f}
   p-value = {p_cu:.4f} {'→ có ý nghĩa thống kê ✅' if p_cu < 0.05 else '→ không có ý nghĩa thống kê ⚠️'}
   Cohen's d = {d_cu:.3f} ({interpret_d(d_cu)} effect size)
""")

print("""3. PHÂN TÍCH TRADE-OFF""")
for model in models_sorted:
    avg_cost = np.mean(cost[model])
    avg_time = np.mean(gen_time[model])
    avg_score = mean_s[model]
    cost_per_point = avg_cost / avg_score if avg_score > 0 else 0
    print(f"   {model:12s}: Score={avg_score:.2f}  Cost=${avg_cost:.3f}  "
          f"Time={avg_time:.0f}s  "
          f"Cost/point=${cost_per_point:.4f}")

print()
print("""4. GỢI Ý CHO BÁO CÁO ĐỒ ÁN
   Copy đoạn này vào phần Kết luận của báo cáo (chỉnh số thực tế):
""")

sig_text = "có ý nghĩa thống kê" if p_cu < 0.05 else "chưa có ý nghĩa thống kê"
print(f"""   ---
   Kết quả thực nghiệm trên {len(FEATURES)} feature cho thấy ChatGPT đạt điểm
   chất lượng trung bình cao nhất ({mean_s['ChatGPT']:.2f}/10), tiếp theo là
   Gemini ({mean_s['Gemini']:.2f}/10) và Qwen open-source ({mean_s['Qwen']:.2f}/10).
   Sự khác biệt giữa mô hình thương mại (ChatGPT) và open-source (Qwen)
   {sig_text} (p={p_cu:.4f}, Cohen's d={d_cu:.3f} — {interpret_d(d_cu)} effect).
   Tuy nhiên, Qwen không phát sinh chi phí và có thể chạy hoàn toàn
   offline, cho thấy đây là lựa chọn khả thi cho môi trường có ràng
   buộc về bảo mật dữ liệu hoặc ngân sách.
   ---""")

print()
print('=' * 70)

## Cell 9 — Xuất kết quả ra CSV & download

In [ ]:
from google.colab import files

# Bảng tổng hợp kết quả
export_rows = []
for model in scores:
    s = np.array(scores[model])
    for i, feature in enumerate(FEATURES):
        export_rows.append({
            'Model': model,
            'Feature': feature,
            'Score_Total': scores[model][i],
            'Score_Coverage': dimension_scores[model][i][0],
            'Score_Writing': dimension_scores[model][i][1],
            'Score_Format': dimension_scores[model][i][2],
            'Score_Domain': dimension_scores[model][i][3],
            'TC_Count': tc_counts[model][i],
            'Gen_Time_s': gen_time[model][i],
            'Cost_USD': cost[model][i],
        })

export_df = pd.DataFrame(export_rows)
export_df.to_csv('llm_evaluation_results.csv', index=False, encoding='utf-8-sig')
print('✅ Đã xuất: llm_evaluation_results.csv')
print(export_df.to_string(index=False))

# Download tất cả files
for fname in ['llm_evaluation_results.csv', 'radar_chart.png',
              'comparison_charts.png']:
    try:
        files.download(fname)
        print(f'📥 Downloading: {fname}')
    except Exception as e:
        print(f'⚠️  {fname}: {e}')